In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
import os

# Đường dẫn nguồn trên Drive
DRIVE_SRC = "/content/drive/MyDrive/HK1-20252026/Steganography/Data/musdb18-hq/train"

print(f" Đang kiểm tra số lượng file tại: {DRIVE_SRC}")

count = 0
for root, dirs, files in os.walk(DRIVE_SRC):
    wav_files = [f for f in files if f.endswith('.wav')]
    count += len(wav_files)
    if count > 0 and count % 100 == 0:
        print(f"   ... Đã tìm thấy {count} file...")

print(f" TỔNG KẾT: Colab nhìn thấy {count} file .wav trên Drive.")

if count < 10:
    print(" CẢNH BÁO: Drive chưa hiển thị đủ file..")

 Đang kiểm tra số lượng file tại: /content/drive/MyDrive/HK1-20252026/Steganography/Data/musdb18-hq/train
 TỔNG KẾT: Colab nhìn thấy 750 file .wav trên Drive.


In [ ]:

!mkdir -p "/content/local_input"
!mkdir -p "/content/local_output"


!cp -r -n -v "/content/drive/MyDrive/HK1-20252026/Steganography/Data/musdb18-hq/train/." "/content/local_input/"

!cp "/content/drive/MyDrive/HK1-20252026/Steganography/Data/audio-cats-and-dogs/5/cat_24.wav" "/content/local_secret.wav"

import os
local_count = sum([len(files) for r, d, files in os.walk("/content/local_input")])
print(f"\n Đã copy xuống máy ảo: {local_count} file.")


 Đã copy xuống máy ảo: 750 file.


In [ ]:

import numpy as np
import os
PROJECT_PATH = '/content/drive/MyDrive/Git-audio-stego/audio-steganograph/AudioStego/exp'

if os.path.exists(PROJECT_PATH):
    os.chdir(PROJECT_PATH)
    !pip install -q librosa soundfile tensorflow scikit-learn seaborn
else:
    print(f" Không tìm thấy thư mục tại {PROJECT_PATH}")




In [ ]:
!python run_case.py \
  --case_id 2 \
  --k 1 \
  --password "NguyenNgocChien" \
  --input_dir "/content/local_input" \
  --secret_file "/content/local_secret.wav" \
  --output_base "/content/local_output"

⏳ Đang quét file (bao gồm thư mục con)...
RUNNING: Case 2: Random Key (Static Salt), Fixed Block | Tìm thấy: 750 files
Pass: NguyenNgocChien | k=1
[49/750] PR - Happy Daze_mixture.wav.../content/drive/MyDrive/Git-audio-stego/audio-steganograph/AudioStego/ablation_study/stego_core.py:53: WavFileWarning: Chunk (non-data) not understood, skipping it.
  rate, audio_data = wavfile.read(cover_path)
/content/drive/MyDrive/Git-audio-stego/audio-steganograph/AudioStego/ablation_study/benchmark_stego.py:15: WavFileWarning: Chunk (non-data) not understood, skipping it.
  r1, y1 = wavfile.read(original_path)
[156/750] PR - Happy Daze_bass.wav.../content/drive/MyDrive/Git-audio-stego/audio-steganograph/AudioStego/ablation_study/stego_core.py:53: WavFileWarning: Chunk (non-data) not understood, skipping it.
  rate, audio_data = wavfile.read(cover_path)
/content/drive/MyDrive/Git-audio-stego/audio-steganograph/AudioStego/ablation_study/benchmark_stego.py:15: WavFileWarning: Chunk (non-data) not under

In [ ]:
import os
import shutil
import time

# --- CẤU HÌNH ---
SOURCE_DIR = "/content/local_output/2_Random_Fixed_DefaultSalt"
# Đường dẫn đích trên Drive (giữ nguyên cấu trúc thư mục)
DEST_DIR = "/content/drive/MyDrive/HK1-20252026/Steganography/Ablation/2_Random_Fixed_DefaultSalt"

print(f" BẮT ĐẦU COPY TỪNG FILE...\nNguồn: {SOURCE_DIR}\nĐích:  {DEST_DIR}")

# Tạo thư mục đích trên Drive
os.makedirs(DEST_DIR, exist_ok=True)

# Lấy danh sách file cần copy
files_to_copy = [f for f in os.listdir(SOURCE_DIR) if f.endswith('.wav') or f.endswith('.csv')]
total_files = len(files_to_copy)
copied_count = 0

start_time = time.time()

for i, filename in enumerate(files_to_copy):
    src_file = os.path.join(SOURCE_DIR, filename)
    dest_file = os.path.join(DEST_DIR, filename)

    # Kiểm tra nếu file đã tồn tại trên Drive với cùng kích thước -> Bỏ qua
    if os.path.exists(dest_file):
        if os.path.getsize(src_file) == os.path.getsize(dest_file):
            print(f"[{i+1}/{total_files}] ⏭ Bỏ qua (Đã có): {filename}")
            copied_count += 1
            continue

    try:
        # Copy file
        shutil.copy2(src_file, dest_file)
        print(f"[{i+1}/{total_files}]  Đã copy: {filename}")
        copied_count += 1

        # Flush bộ nhớ đệm xuống đĩa để đảm bảo file được ghi thật sự
        if (i + 1) % 10 == 0:
            os.sync()

    except OSError as e:
        if e.errno == 28: # Lỗi hết dung lượng
            print(f"\n DỪNG LẠI: Google Drive ĐÃ HẾT DUNG LƯỢNG (Full) tại file thứ {i+1}.")
            print(f"File gây lỗi: {filename}")
            print(f"Kích thước file này: {os.path.getsize(src_file) / (1024*1024):.2f} MB")
            break
        else:
            print(f"\n Lỗi khi copy file {filename}: {e}")

print(f"\n KẾT THÚC QUÁ TRÌNH.")
print(f"Tổng số file đã xử lý: {copied_count}/{total_files}")
print(f"Thời gian chạy: {time.time() - start_time:.2f} giây")

🚀 BẮT ĐẦU COPY TỪNG FILE...
Nguồn: /content/local_output/2_Random_Fixed_DefaultSalt
Đích:  /content/drive/MyDrive/HK1-20252026/Steganography/Ablation/2_Random_Fixed_DefaultSalt
[1/750] ✅ Đã copy: Lushlife - Toynbee Suite_drums.wav
[2/750] ✅ Đã copy: M.E.R.C. Music - Knockout_mixture.wav
[3/750] ✅ Đã copy: Jokers, Jacks & Kings - Sea Of Leaves_mixture.wav
[4/750] ✅ Đã copy: Young Griffo - Facade_drums.wav
[5/750] ✅ Đã copy: Creepoid - OldTree_other.wav
[6/750] ✅ Đã copy: Leaf - Summerghost_vocals.wav
[7/750] ✅ Đã copy: Celestial Shore - Die For Us_bass.wav
[8/750] ✅ Đã copy: Grants - PunchDrunk_mixture.wav
[9/750] ✅ Đã copy: Secretariat - Borderline_other.wav
[10/750] ✅ Đã copy: Music Delta - Country1_mixture.wav
[11/750] ✅ Đã copy: Music Delta - Reggae_other.wav
[12/750] ✅ Đã copy: Skelpolu - Together Alone_other.wav
[13/750] ✅ Đã copy: Johnny Lokke - Whisper To A Scream_vocals.wav
[14/750] ✅ Đã copy: Traffic Experiment - Once More (With Feeling)_other.wav
[15/750] ✅ Đã copy: The Distr

In [ ]:
import os

# Đường dẫn thư mục trên Drive cần kiểm tra
target_dir = "/content/drive/MyDrive/HK1-20252026/Steganography/Ablation/2_Random_Fixed_DefaultSalt"

print(f"Đang kiểm tra thư mục: {target_dir}")

if os.path.exists(target_dir):
    # Lấy danh sách tất cả file .wav
    files = [f for f in os.listdir(target_dir) if f.endswith('.wav')]
    count = len(files)

    print(f"Tổng số file .wav hiện có: {count}")

    if count >= 750:
        print("Đã đủ 750 file.")
    else:
        print(f"Chưa đủ. Còn thiếu: {750 - count} file.")
else:
    print("Thư mục chưa được tạo hoặc đường dẫn sai.")

Đang kiểm tra thư mục: /content/drive/MyDrive/HK1-20252026/Steganography/Ablation/2_Random_Fixed_DefaultSalt
Tổng số file .wav hiện có: 750
Đã đủ 750 file.


In [ ]:
!python run_case.py \
  --case_id 1 \
  --input_dir "/content/local_input" \
  --secret_file "/content/local_secret.wav" \
  --output_base "/content/drive/MyDrive/HK1-20252026/Steganography/Ablation"

[*] Case: 1_NoRandom | Input: local_input
[*] Secret File: /content/local_secret.wav
[*] Log path: logs/benchmark_local_input_1_NoRandom_20260201_040937.csv
[*] Found: 750 files
[750/750] Zeno - Signs_vocals.wav -> PSNR: 107.36 dB
[DONE] Hoàn tất. Log tại: logs/benchmark_local_input_1_NoRandom_20260201_040937.csv
